# AMFCA Master Reproducibility Colab

This notebook is the single entry point for the consolidated repository. It preserves the frozen Lhende 2026 workflows and exposes a configuration-driven runner.

**Important:** Earth Engine stages can be expensive and require authentication. Run the plan and repository verification before any heavy stage.

In [ ]:
#@title 1. Get the repository
REPO_URL = "https://github.com/romenmeitei/AMFCA.git" #@param {type:"string"}
REPO_DIR = "/content/AMFCA"

from pathlib import Path
import os, subprocess
    print("Replace REPO_URL after uploading the repository to GitHub. If you uploaded the ZIP to Colab instead, unzip it to /content/AMFCA.")
else:
    if not Path(REPO_DIR).exists():
        subprocess.run(["git","clone",REPO_URL,REPO_DIR],check=True)
os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())

In [ ]:
#@title 2. Install AMFCA and mount Drive
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q","-e",".[earthengine,dev]"],check=True)
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
#@title 3. Verify repository (offline)
import subprocess, sys
subprocess.run([sys.executable,"scripts/verify_repository.py"],check=True)

In [ ]:
#@title 4. Choose configuration and inspect the plan
CONFIG = "configs/lhende_2026.yaml" #@param {type:"string"}
subprocess.run([sys.executable,"scripts/run_pipeline.py","--config",CONFIG,"--plan"],check=True)

## Optional Stage 0 — render the frozen Earth Engine core

For exact Lhende reproduction, this creates a configured COPY; it never edits the frozen script. Review `configs/lhende_2026.yaml` first. If you already have the exported core products, skip this stage.

In [ ]:
#@title 5. Render (but do not automatically execute) the core GEE script
RENDER_CORE = False #@param {type:"boolean"}
if RENDER_CORE:
    subprocess.run([sys.executable,"scripts/render_core_from_config.py","--config",CONFIG,"--output","/content/lhende_core_configured.py"],check=True)
    print("Rendered: /content/lhende_core_configured.py")
    print("Inspect it, then execute with: %run /content/lhende_core_configured.py")

## Downstream reproducibility pipeline

The recommended approach is stage-by-stage execution. If a stage was already completed, put its directory in the corresponding `existing_*` field in the YAML and begin from a later stage.

In [ ]:
#@title 6. Choose downstream stages
STAGES = [
    "m01_concurrence",
    "background_comparison",
    "patch_inventory",
    "s2_recovery",
    "directional",
    "dated_scene_audit",
    "inventory_reconciliation",
    "top20_review",
    "external_corroboration",
]
RUN_PIPELINE = False #@param {type:"boolean"}
if RUN_PIPELINE:
    cmd=[sys.executable,"scripts/run_pipeline.py","--config",CONFIG,"--stages",*STAGES]
    subprocess.run(cmd,check=True)
else:
    print("Dry setup only. Set RUN_PIPELINE=True when inputs and Earth Engine access are ready.")

## Generic future-study mode

Copy `configs/template_new_event.yaml`, point it to an exactly co-registered SAR control-pattern raster, spectral-RMS raster and zone GeoJSON, and run the same CLI. Generic mode refuses mismatched grids instead of resampling them.